# Analisi della serie storica GS10 (rendimento dei Treasury USA a 10 anni)

**Dati.** `GS10` (FRED): rendimento dei titoli di Stato USA a 10 anni a scadenza costante, **mensile**, in **punti percentuali**, gennaio 2000 – ottobre 2026 ($T = 322$). Ogni valore mensile è la **media dei giorni lavorativi** del mese: questo dettaglio avrà un ruolo importante nella scelta del modello (sezione 4).

**Obiettivo.** Costruire un modello ARIMA per la serie, verificarne la bontà e confrontarne le previsioni con quelle di un random walk.

**Procedura** (ogni passo usa le classi di `tests.py` e `models.py`):

1. Dati, scelta della trasformazione (livelli o logaritmi) e distribuzione.
2. Stazionarietà: ADF e KPSS, poi test sul drift → scelta di $d$ e della costante.
3. Ljung-Box sulla serie differenziata: c'è qualcosa da modellare?
4. Identificazione: ACF (bande di Bartlett) e PACF → candidati per $p$ e $q$.
5. Selezione dell'ordine con AIC e BIC.
6. Stima (Hannan-Rissanen), t-test, radici, test sull'effetto di media mensile.
7. Diagnostica dei residui: Ljung-Box, ACF, Jarque-Bera.
8. Stabilità dei parametri.
9. Valutazione fuori campione: RMSE, MAE, copertura degli intervalli, Diebold-Mariano.
10. Previsione a 12 mesi.
11. Controllo dell'implementazione con `statsmodels`.

**Modifiche rispetto alla versione precedente del codice** (dettagli nel report):
- `LjungBox_test`: il valore critico era una probabilità (`chi2.cdf`) confrontata con la statistica; ora è il quantile $\chi^2_{dof,0.95}$, con i gradi di libertà $m-p-q$ sui residui.
- `JarqueBera_test`: usava una variabile globale e chiavi inesistenti; ora lavora sui residui e restituisce anche il p-value.
- `ARMA`: la parte MA era stimata regredendo i residui sui propri ritardi (cioè un AR sui residui, non un MA) e senza costante; ora è il metodo di Hannan-Rissanen (due OLS), con costante opzionale, shock ricostruiti in modo ricorsivo, radici, pesi $\psi$ e previsione.
- `ACF_test`: bande di Bartlett al posto di $\pm1.96/\sqrt T$. `ACF_test`/`PACF_test`: il suggerimento è il primo ritardo **non** significativo meno 1, non l'ultimo significativo su 20 (che con 20 test al 5% trova quasi sempre un ritardo "significativo" per caso: da qui il $p = 15$).
- `ADF_test`: AIC calcolato sullo stesso campione per tutti i ritardi, versioni `'c'` e `'ct'`, valori critici di MacKinnon per il $T$ effettivo. `KPSS_test`: versioni `'c'` e `'ct'`.
- Nuovi: `Trend_test` (drift/trend con errori standard HAC), `T_test`, `Coefficient_test`, `Stability_test`, `DieboldMariano_test`, classe `ARIMA`, `RandomWalk` con drift e previsione, metriche `rmse`, `mae`, `coverage`, `aic`, `bic`, `long_run_variance`.

In [1]:
import json
import os
import warnings

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from dataset import load
from metrics import aic, bic, coverage, kurtosis, mae, rmse, skewness
from models import ARIMA, ARMA, RandomWalk, rolling_one_step
from tests import (ACF_test, ADF_test, Coefficient_test, DieboldMariano_test, DW_test,
                   JarqueBera_test, KPSS_test, LjungBox_test, PACF_test, Stability_test,
                   T_test, Trend_test)

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:.4f}".format
os.makedirs("figures", exist_ok=True)
R = {}          # risultati salvati in results.json (usati nel report LaTeX)

## 1. Dati e trasformazione

In [2]:
s = load()
dates = s.index
y = s.values                 # livello del rendimento, in %
dy = np.diff(y)              # variazione mensile, in punti percentuali
T = len(y)

print(f"Periodo: {dates[0]:%Y-%m} - {dates[-1]:%Y-%m}, T = {T}")
print(f"Minimo {y.min():.2f}% ({dates[y.argmin()]:%Y-%m}), massimo {y.max():.2f}% ({dates[y.argmax()]:%Y-%m}), ultimo {y[-1]:.2f}%")

fig, axes = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
axes[0].plot(dates, y, label="GS10 (livello, %)")
axes[0].set_title("Rendimento del Treasury a 10 anni")
axes[0].set_ylabel("%")
axes[0].legend()
axes[1].plot(dates[1:], dy, color="tab:green", label="Variazione mensile $\\Delta y_t$ (p.p.)")
axes[1].axhline(0.0, color="black", linewidth=0.8)
axes[1].set_title("Variazioni mensili")
axes[1].set_ylabel("punti percentuali")
axes[1].legend()
fig.tight_layout()
fig.savefig("figures/serie.png", dpi=150)
plt.show()

R["T"] = T
R["start"], R["end"] = f"{dates[0]:%Y-%m}", f"{dates[-1]:%Y-%m}"
R["ymin"], R["ymin_date"] = float(y.min()), f"{dates[y.argmin()]:%Y-%m}"
R["ymax"], R["ymax_date"] = float(y.max()), f"{dates[y.argmax()]:%Y-%m}"
R["ylast"] = float(y[-1])

Periodo: 2000-01 - 2026-10, T = 322
Minimo 0.62% (2020-07), massimo 6.66% (2000-01), ultimo 5.13%


### 1.1 Livelli o logaritmi?

GS10 **non è un prezzo**, è un tasso d'interesse in percentuale. La differenza dei logaritmi, $\Delta\log y_t$, non è un "rendimento": è la variazione **percentuale del tasso**. Quando il tasso è basso (nel 2020 è sceso allo 0,62%), la stessa variazione di 0,1 punti vale il 16% in logaritmi, mentre con il tasso al 5% vale il 2%. Se la variabilità delle variazioni in punti è circa costante, quella in logaritmi sarà quindi molto più alta negli anni di tassi bassi.

Poiché nel corso **non** si modella la varianza che cambia nel tempo (niente GARCH), conviene scegliere la trasformazione con la **varianza più stabile**. Confronto la deviazione standard delle due serie di differenze in quattro sottoperiodi e su una finestra mobile di 36 mesi.

In [3]:
dlog = np.diff(np.log(y))
periods = [("2000-2007", 2000, 2007), ("2008-2014", 2008, 2014), ("2015-2021", 2015, 2021), ("2022-2026", 2022, 2026)]
years = dates[1:].year
rows = []
for name, a, b in periods:
    m = (years >= a) & (years <= b)
    rows.append([name, y[1:][m].mean(), dy[m].std(ddof=1), dlog[m].std(ddof=1)])
tab_var = pd.DataFrame(rows, columns=["periodo", "livello medio (%)", "sd Δy (p.p.)", "sd Δlog y"])
ratio_dy = tab_var["sd Δy (p.p.)"].max() / tab_var["sd Δy (p.p.)"].min()
ratio_dl = tab_var["sd Δlog y"].max() / tab_var["sd Δlog y"].min()
display(tab_var)
print(f"Rapporto sd massima / minima:  Δy = {ratio_dy:.2f}   Δlog y = {ratio_dl:.2f}")

w = 36
roll_dy = pd.Series(dy, index=dates[1:]).rolling(w).std()
roll_dl = pd.Series(dlog, index=dates[1:]).rolling(w).std()
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(roll_dy / roll_dy.mean(), label="$\\Delta y$ (livelli)")
ax.plot(roll_dl / roll_dl.mean(), label="$\\Delta \\log y$ (logaritmi)")
ax.axhline(1.0, color="black", linewidth=0.8)
ax.set_title("Deviazione standard mobile a 36 mesi (divisa per la sua media)")
ax.legend()
fig.tight_layout()
fig.savefig("figures/varianza_mobile.png", dpi=150)
plt.show()

R["tab_var"] = tab_var.values.tolist()
R["ratio_dy"], R["ratio_dl"] = float(ratio_dy), float(ratio_dl)

,periodo,livello medio (%),sd Δy (p.p.),sd Δlog y
0,2000-2007,4.6866,0.2179,0.0485
1,2008-2014,2.8025,0.2345,0.0849
2,2015-2021,1.9569,0.1676,0.1080
3,2022-2026,3.9684,0.2230,0.0674


Rapporto sd massima / minima:  Δy = 1.40   Δlog y = 2.23


**Scelta.** Nelle variazioni in punti percentuali la deviazione standard resta tra circa 0,17 e 0,23 punti in tutti i sottoperiodi; in logaritmi più che raddoppia negli anni dei tassi vicini allo zero. Lavoro quindi sui **livelli**: $y_t$ è il rendimento in %, $\Delta y_t$ la variazione mensile in punti percentuali. Il resto dell'analisi usa queste due serie.

Distribuzione delle variazioni mensili:

In [4]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
mu, sigma = np.mean(dy), np.std(dy)
sk, ku = skewness(dy), kurtosis(dy)
ax.hist(dy, bins=40, density=True, alpha=0.65, color="tab:green", edgecolor="black", label="Empirica")
x = np.linspace(dy.min(), dy.max(), 300)
ax.plot(x, stats.norm.pdf(x, mu, sigma), "r--", linewidth=2, label="Normale $\\mathcal{N}(\\mu, \\sigma^2)$")
ax.axvline(mu, color="black", linestyle=":", linewidth=1.2, label=f"Media = {mu:.4f}")
ax.set_title("Distribuzione di $\\Delta y_t$")
ax.set_xlabel("punti percentuali")
ax.legend(loc="upper right")
ax.text(0.03, 0.95, f"Skewness: {sk:.4f}\nKurtosis: {ku:.4f}", transform=ax.transAxes, va="top",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85))
fig.tight_layout()
fig.savefig("figures/distribuzione.png", dpi=150)
plt.show()

R["dy_mean"], R["dy_sd"], R["dy_skew"], R["dy_kurt"] = float(mu), float(np.std(dy, ddof=1)), sk, ku

Le variazioni hanno media quasi nulla, leggera asimmetria negativa e **curtosi sopra 3** (code più spesse della normale): mesi con variazioni molto grandi, come il 2008 e il 2022, capitano più spesso di quanto preveda una normale. Se ne terrà conto nel giudicare gli intervalli di previsione (sezione 7).

## 2. Stazionarietà: scelta di $d$

**Perché ADF e KPSS insieme.** I due test hanno le ipotesi invertite (ADF: H0 radice unitaria; KPSS: H0 stazionarietà), e ciascuno accetta H0 finché i dati non la smentiscono. Se concordano la conclusione è solida; se discordano, i dati non bastano o c'è una rottura.

**Quale versione.** Sui **livelli** uso entrambe le versioni: `'ct'` ammette un trend deterministico (la serie scende fino al 2020 e risale dopo), `'c'` una media costante. Sulle **differenze** uso solo `'c'`: un trend lineare nei livelli diventerebbe una costante nelle differenze, non un trend.

**Regole di decisione.** ADF: rifiuto se $\tau$ è **minore** del valore critico (coda sinistra; valori critici di MacKinnon per il $T$ effettivo). KPSS: rifiuto se $\eta$ è **maggiore** del valore critico (coda destra). I ritardi dell'ADF sono scelti con l'AIC su un campione comune; il KPSS usa $L = \lfloor 4(T/100)^{1/4}\rfloor$.

In [5]:
rows = []
for label, series, regs in [("livello y", y, ["c", "ct"]), ("Δy", dy, ["c"])]:
    for reg in regs:
        adf = ADF_test(series, regression=reg)
        a_stat, a_ok, _ = adf.test()
        kp = KPSS_test(series, regression=reg)
        k_stat, k_ok, _ = kp.test()
        rows.append([label, reg, a_stat, adf.lag, adf.crit[0.05], "stazionaria" if a_ok else "radice unitaria",
                     k_stat, kp.crit[0.05], "stazionaria" if k_ok else "non stazionaria"])
tab_stat = pd.DataFrame(rows, columns=["serie", "versione", "ADF τ", "lag", "crit 5%", "ADF dice",
                                       "KPSS η", "crit 5%", "KPSS dice"])
display(tab_stat)
R["tab_stat"] = tab_stat.values.tolist()

,serie,versione,ADF τ,lag,crit 5%,ADF dice,KPSS η,crit 5%,KPSS dice
0,livello y,c,-2.1940,2,-2.8706,radice unitaria,2.0734,0.4630,non stazionaria
1,livello y,ct,-1.4528,2,-3.4243,radice unitaria,0.8619,0.1460,non stazionaria
2,Δy,c,-12.3636,1,-2.8706,stazionaria,0.4200,0.4630,stazionaria


**Livelli.** In entrambe le versioni l'ADF non rifiuta la radice unitaria e il KPSS rifiuta la stazionarietà: i due test concordano, **$y_t$ non è stazionaria**, né attorno a una media né attorno a un trend.

**Differenze.** L'ADF rifiuta nettamente la radice unitaria e il KPSS non rifiuta la stazionarietà: **$\Delta y_t$ è stazionaria**. Quindi $d = 1$ e non serve una seconda differenza.

### 2.1 C'è un drift?

Con $d=1$ la costante del modello è il **drift**, cioè la variazione media mensile. Lo verifico con un t-test sulla media di $\Delta y_t$, con errori standard di **Newey-West** perché $\Delta y_t$ è autocorrelata.

Non si fa invece il t-test sul trend nei livelli ($y_t = a + bt + u_t$): con una radice unitaria quel t non segue la distribuzione normale ed è artificialmente grande (regressione spuria). Lo calcolo solo per mostrarlo.

In [6]:
t_d, p_d, sig_d, verdict_d = Trend_test(dy, kind="drift").test()
print("Drift su Δy        :", verdict_d)
t_tr, p_tr, _, verdict_tr = Trend_test(y, kind="trend").test()
print("Trend sui livelli  :", verdict_tr, " <- NON valido: y ha una radice unitaria")

R["drift"] = [float(np.mean(dy)), t_d, p_d]
R["trend_spurio"] = [t_tr, p_tr]

Drift su Δy        : drift = -0.00477 (SE HAC 0.01340), t = -0.356, p = 0.722: drift non significativo.
Trend sui livelli  : trend = -0.00648 (SE HAC 0.00190), t = -3.414, p = 0.001: trend significativo.  <- NON valido: y ha una radice unitaria


Il drift stimato è circa $-0{,}005$ punti al mese, con $t \approx -0{,}36$: **non significativo**. Il modello non avrà costante (ARIMA senza drift), coerentemente con il fatto che un tasso d'interesse non può crescere o calare all'infinito. Il t del trend sui livelli sembrerebbe significativo, ma è il classico risultato spurio di una regressione su una serie con radice unitaria, ed è proprio per questo che il drift si testa sulle differenze.

## 3. C'è qualcosa da modellare?

Se $\Delta y_t$ fosse rumore bianco, il modello migliore sarebbe il random walk ($\hat y_{T+1} = y_T$) e non servirebbe un ARMA. Lo verifico con il **Ljung-Box** su $\Delta y_t$ con $m = 6$ e $m = 12$ ritardi; i gradi di libertà sono $m$ perché non è stato stimato nessun parametro ARMA.

Riporto anche il **Durbin-Watson** sui residui del random walk (che sono proprio le $\Delta y_t$), solo come misura descrittiva: $d \approx 2(1-\hat\rho_1)$.

In [7]:
rows = []
for m in (6, 12):
    q, p, ok = LjungBox_test(dy, lags=m).test()
    rows.append([m, m, q, stats.chi2.ppf(0.95, m), p, "rumore bianco" if ok else "autocorrelazione"])
tab_lb0 = pd.DataFrame(rows, columns=["m", "dof", "Q", "crit 5%", "p-value", "esito"])
display(tab_lb0)

rw = RandomWalk()
rw.train(y)
dw_rw, _, _ = DW_test(rw).test()
print(f"DW sui residui del random walk = {dw_rw:.4f}  ->  rho_1 ≈ 1 - DW/2 = {1 - dw_rw / 2:.3f}")

R["tab_lb0"] = tab_lb0.values.tolist()
R["dw_rw"] = dw_rw

,m,dof,Q,crit 5%,p-value,esito
0,6,6,22.7554,12.5916,0.0009,autocorrelazione
1,12,12,27.4494,21.0261,0.0067,autocorrelazione


DW sui residui del random walk = 1.5135  ->  rho_1 ≈ 1 - DW/2 = 0.243


Il Ljung-Box rifiuta: **le variazioni mensili sono autocorrelate**, quindi un modello ARMA per $\Delta y_t$ può migliorare le previsioni rispetto al random walk. Il DW di circa 1,5 indica un'autocorrelazione positiva al primo ritardo di circa 0,24.

## 4. Identificazione: ACF e PACF di $\Delta y_t$

- **ACF** con bande di **Bartlett**: al ritardo $k$ la banda è $\pm 1{,}96\sqrt{(1 + 2\sum_{j<k}\hat\rho_j^2)/T}$, cioè il test di H0 "MA($k-1$)". Il suggerimento per $q$ è il primo ritardo non significativo meno 1.
- **PACF** con banda $\pm 1{,}96/\sqrt T$ (test di H0 "AR($k-1$)"). Il suggerimento per $p$ è il primo ritardo non significativo meno 1.

**Perché non "l'ultimo ritardo significativo".** Guardando 20 ritardi, ciascuno con il 5% di probabilità di uscire dalle bande per caso, la probabilità che almeno uno esca è $1 - 0{,}95^{20} \approx 64\%$. Nella versione precedente la regola "ultimo ritardo significativo" aveva preso una barra casuale al ritardo 15 e portato a un ARMA(15, 1).

In [8]:
acf = ACF_test(dy, nlags=24, save_path="figures/acf_dy.png", title="ACF di $\\Delta y_t$ (bande di Bartlett)")
q_sugg, acf_path = acf.test()
pacf = PACF_test(dy, nlags=24, save_path="figures/pacf_dy.png", title="PACF di $\\Delta y_t$")
p_sugg, pacf_path = pacf.test()

print("ACF  ritardi 1-6:", np.round(acf.values[1:7], 3))
print("PACF ritardi 1-6:", np.round(pacf.values[1:7], 3), f"(banda ±{1.96 / np.sqrt(len(dy)):.3f})")
print(f"q suggerito dall'ACF = {q_sugg},  p suggerito dalla PACF = {p_sugg}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].imshow(mpimg.imread(acf_path)); axes[0].axis("off")
axes[1].imshow(mpimg.imread(pacf_path)); axes[1].axis("off")
fig.tight_layout()
plt.show()

R["acf"] = acf.values[1:7].tolist()
R["pacf"] = pacf.values[1:7].tolist()
R["q_sugg"], R["p_sugg"] = q_sugg, p_sugg
R["pacf_band"] = 1.96 / np.sqrt(len(dy))

ACF  ritardi 1-6: [ 0.242 -0.055 -0.004 -0.042 -0.072 -0.042]
PACF ritardi 1-6: [ 0.242 -0.12   0.042 -0.064 -0.045 -0.021] (banda ±0.109)
q suggerito dall'ACF = 1,  p suggerito dalla PACF = 2


**Lettura.**
- L'ACF ha una sola barra significativa, al ritardo 1 ($\hat\rho_1 \approx 0{,}24$), e poi **si interrompe**: è la firma di un **MA(1)**.
- La PACF ha 0,24 al ritardo 1 e circa $-0{,}12$ al ritardo 2, poi resta nelle bande: decresce con segno alterno, come succede proprio in un MA(1) con $\theta > 0$. Presa alla lettera suggerisce anche un AR(2).
- $\hat\rho_1 > 0$: la differenziazione non è eccessiva (con una differenza di troppo $\hat\rho_1$ sarebbe vicino a $-0{,}5$).

Candidati: **MA(1)**, AR(2) e i modelli vicini, da confrontare con AIC e BIC.

**Perché un MA(1): l'effetto della media mensile.** Ogni dato di GS10 è la media dei valori giornalieri del mese. Se il tasso giornaliero fosse un random walk puro, le variazioni delle **medie mensili** non sarebbero indipendenti: due mesi consecutivi condividono gli shock arrivati dentro il mese precedente, che pesano su entrambe le medie. Si dimostra (Working, 1960) che per una media su molti giorni l'autocorrelazione al primo ritardo delle differenze è $\rho_1 = 1/4$, e zero dal secondo ritardo in poi: esattamente un MA(1). Il $\theta$ corrispondente risolve $\theta/(1+\theta^2) = 1/4$:
$$\theta^2 - 4\theta + 1 = 0 \quad\Rightarrow\quad \theta = 2 - \sqrt 3 \approx 0{,}268.$$
L'autocorrelazione osservata (0,24) è molto vicina a 0,25: il modello atteso per una media mensile di un random walk è proprio un ARIMA(0, 1, 1) con $\theta \approx 0{,}27$. Lo verifico nella sezione 6.

## 5. Selezione dell'ordine con AIC e BIC

Stimo tutti gli ARMA($p$, $q$) con $p, q \le 3$ su $\Delta y_t$ (senza costante), più l'ARMA(15, 1) della versione precedente. Per confrontarli correttamente tutti i criteri sono calcolati **sulle stesse osservazioni** (scartando le prime 16, cioè il ritardo massimo più uno):
$$AIC = \log\frac{RSS}{n} + \frac{2k}{n}, \qquad BIC = \log\frac{RSS}{n} + \frac{k\log n}{n}.$$
Il BIC penalizza di più i parametri ed è il criterio principale; l'AIC serve da controllo.

In [9]:
burn = 16
rows = []
orders = [(p, q) for p in range(4) for q in range(4)] + [(15, 1)]
for p, q in orders:
    m = ARMA(p, q, const=False)
    m.train(dy)
    a, b = m.information_criteria(burn)
    rows.append([f"ARMA({p},{q})", p + q, a, b, m.is_stationary(), m.is_invertible()])
tab_ic = pd.DataFrame(rows, columns=["modello", "k", "AIC", "BIC", "stazionario", "invertibile"])
tab_ic = tab_ic.sort_values("BIC").reset_index(drop=True)
display(tab_ic.head(8))
display(tab_ic[tab_ic["modello"] == "ARMA(15,1)"])

best = tab_ic.iloc[0]["modello"]
print("Minimo BIC:", best, "| Minimo AIC:", tab_ic.sort_values("AIC").iloc[0]["modello"])
R["tab_ic"] = tab_ic.values.tolist()

,modello,k,AIC,BIC,stazionario,invertibile
0,"ARMA(0,1)",1,-3.1951,-3.1829,True,True
1,"ARMA(0,2)",2,-3.1942,-3.1698,True,True
2,"ARMA(1,1)",2,-3.1883,-3.1639,True,True
3,"ARMA(2,0)",2,-3.1879,-3.1635,True,True
4,"ARMA(1,0)",1,-3.1745,-3.1623,True,True
5,"ARMA(1,2)",3,-3.1884,-3.1518,True,True
6,"ARMA(0,3)",3,-3.1871,-3.1505,True,True
7,"ARMA(2,1)",3,-3.1845,-3.1479,True,True


,modello,k,AIC,BIC,stazionario,invertibile
16,"ARMA(15,1)",16,-3.1201,-2.9250,True,True


Minimo BIC: ARMA(0,1) | Minimo AIC: ARMA(0,1)


L'**MA(1)** ha sia il BIC sia l'AIC più bassi. Gli altri modelli in cima alla classifica (MA(2), ARMA(1,1), AR(2)) aggiungono un parametro senza migliorare l'adattamento abbastanza da compensarlo. L'ARMA(15, 1) della versione precedente è tra i peggiori per il BIC: 16 parametri per catturare quello che un solo $\theta$ descrive già.

**Modello scelto: ARIMA(0, 1, 1) senza costante**
$$\Delta y_t = \varepsilon_t + \theta\,\varepsilon_{t-1}.$$

## 6. Stima e controllo dei coefficienti

**Stima (Hannan-Rissanen).** Gli shock $\varepsilon_{t-1}$ non si osservano, quindi non si può fare direttamente OLS. Si procede in due passi:
1. si stima un AR lungo (ordine $m = \lfloor(\log T)^2\rfloor$) e si prendono i suoi residui come stima degli shock;
2. si fa OLS di $\Delta y_t$ sui ritardi delle stime degli shock.

Gli errori standard vengono da $\hat\sigma^2(X'X)^{-1}$ della seconda regressione. Le stime di massima verosimiglianza di `statsmodels` servono da confronto (sezione 11).

**Controlli.** t-test su $\theta$; invertibilità (la radice di $1 + \theta z$ deve avere $|z| > 1$); z-test di $H_0: \theta = 2 - \sqrt 3$ (effetto della media mensile).

In [10]:
model = ARIMA(0, 1, 1, const=False)
model.train(y)
st = model.get_stats()
theta, se_theta = model.beta[0], st["std_errors"][0]
t_vals, p_vals, signif = T_test(model).test()
ar_roots, ma_roots = model.roots()

print(f"theta = {theta:.4f}  (SE {se_theta:.4f}),  t = {t_vals[0]:.3f},  p = {p_vals[0]:.2e}")
print(f"sigma (deviazione standard degli shock) = {np.sqrt(model.sigma2):.4f} punti percentuali")
print(f"Radice MA: {ma_roots[0]:.3f}  ->  |z| = {abs(ma_roots[0]):.3f} > 1: invertibile = {model.arma.is_invertible()}")

theta0 = 2 - np.sqrt(3)
z_w, p_w, ok_w = Coefficient_test(theta, se_theta, theta0).test()
print(f"H0: theta = 2 - sqrt(3) = {theta0:.4f}:  z = {z_w:.3f}, p = {p_w:.3f}  ->  non rifiutata = {ok_w}")

R["theta"], R["se_theta"], R["t_theta"], R["p_theta"] = float(theta), float(se_theta), float(t_vals[0]), float(p_vals[0])
R["sigma"] = float(np.sqrt(model.sigma2))
R["ma_root"] = float(abs(ma_roots[0]))
R["working"] = [float(theta0), z_w, p_w]

theta = 0.3053  (SE 0.0621),  t = 4.918,  p = 1.48e-06
sigma (deviazione standard degli shock) = 0.2054 punti percentuali
Radice MA: -3.276  ->  |z| = 3.276 > 1: invertibile = True
H0: theta = 2 - sqrt(3) = 0.2679:  z = 0.602, p = 0.547  ->  non rifiutata = True


- $\hat\theta \approx 0{,}31$ è **molto significativo**: la componente MA non è un artefatto.
- La radice MA è circa $-3{,}3$, fuori dal cerchio unitario: il modello è **invertibile** (gli shock passati si ricostruiscono dai dati).
- Il test di $\theta = 2-\sqrt3$ **non rifiuta**: la dinamica stimata è compatibile con quella prodotta dalla sola media mensile di un random walk giornaliero.

## 7. Diagnostica dei residui

Se il modello è corretto, i residui $\hat\varepsilon_t$ sono rumore bianco.

- **Ljung-Box** con $m = 6, 12, 24$ e gradi di libertà $m - p - q = m - 1$ (un parametro stimato "consuma" un grado di libertà).
- **ACF dei residui**, per vedere se resta autocorrelazione a qualche ritardo.
- **Jarque-Bera** sulla normalità: non cambia le previsioni puntuali, ma dice se gli intervalli $\pm1{,}96\hat\sigma$ sono affidabili.
- Il **Durbin-Watson** è riportato solo come indicazione: non è valido per modelli con dinamica stimata, e il Ljung-Box lo sostituisce.

In [11]:
res = model.innovations[1:]      # il primo shock e' posto a 0 dalla ricorsione

rows = []
for m in (6, 12, 24):
    q, p, ok = LjungBox_test(res, lags=m, dof=m - 1).test()
    rows.append([m, m - 1, q, stats.chi2.ppf(0.95, m - 1), p, "rumore bianco" if ok else "autocorrelazione"])
tab_lb = pd.DataFrame(rows, columns=["m", "dof", "Q", "crit 5%", "p-value", "esito"])
display(tab_lb)

jb, p_jb, ok_jb, verdict_jb = JarqueBera_test(res).test()
print("Jarque-Bera:", verdict_jb)

dw_arima = float(np.sum(np.diff(res) ** 2) / (res @ res))
print(f"DW sui residui (indicativo) = {dw_arima:.4f}")

acf_res = ACF_test(res, nlags=24, save_path="figures/acf_residui.png", title="ACF dei residui dell'ARIMA(0,1,1)")
acf_res.test()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(dates[2:], res, color="tab:orange", linewidth=0.9)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Residui dell'ARIMA(0,1,1)")
axes[0].set_ylabel("punti percentuali")
stats.probplot(res / res.std(), dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot dei residui standardizzati")
fig.tight_layout()
fig.savefig("figures/residui.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ax.imshow(mpimg.imread("figures/acf_residui.png")); ax.axis("off")
plt.show()

R["tab_lb"] = tab_lb.values.tolist()
R["jb"] = [jb, p_jb, skewness(res), kurtosis(res)]
R["dw_arima"] = dw_arima
R["acf_res_max"] = float(np.max(np.abs(acf_res.values[1:])))

,m,dof,Q,crit 5%,p-value,esito
0,6,5,2.9767,11.0705,0.7036,rumore bianco
1,12,11,7.0723,19.6751,0.7932,rumore bianco
2,24,23,13.9546,35.1725,0.9282,rumore bianco


Jarque-Bera: JB=64.4764 (crit=5.9915, p=0.0000, S=-0.242, K=5.145) -> normali=False
DW sui residui (indicativo) = 2.0543


- **Ljung-Box**: non rifiuta per nessun $m$. Il singolo $\theta$ ha eliminato l'autocorrelazione che c'era in $\Delta y_t$ (DW ora vicino a 2). Il modello cattura tutta la dinamica lineare.
- **Jarque-Bera**: rifiuta la normalità a causa della curtosi (code spesse, si vede nel Q-Q plot alle estremità). Le stime e le previsioni puntuali restano valide; gli intervalli al 95% basati sulla normale potrebbero essere un po' troppo stretti nei mesi estremi. La copertura effettiva è verificata fuori campione nella sezione 9.

## 8. Stabilità del parametro

Un buon modello di previsione deve avere parametri che non cambiano nel tempo. Stimo l'ARIMA(0, 1, 1) separatamente sulla prima e sulla seconda metà del campione (che contiene la crisi del 2008, i tassi vicini allo zero e il rialzo del 2022) e confronto i $\theta$ con
$$z = \frac{\hat\theta_1 - \hat\theta_2}{\sqrt{SE_1^2 + SE_2^2}}.$$

In [12]:
split = T // 2
b1, se1, b2, se2, z_st, p_st, stable = Stability_test(lambda: ARIMA(0, 1, 1, const=False), y, split).test()
print(f"Prima metà  ({dates[0]:%Y-%m} - {dates[split - 1]:%Y-%m}): theta = {b1[0]:.4f} (SE {se1[0]:.4f})")
print(f"Seconda metà ({dates[split]:%Y-%m} - {dates[-1]:%Y-%m}): theta = {b2[0]:.4f} (SE {se2[0]:.4f})")
print(f"z = {z_st[0]:.3f}, p = {p_st[0]:.3f}  ->  parametro stabile = {stable}")

R["stab"] = [f"{dates[split - 1]:%Y-%m}", f"{dates[split]:%Y-%m}", float(b1[0]), float(se1[0]),
             float(b2[0]), float(se2[0]), float(z_st[0]), float(p_st[0])]

Prima metà  (2000-01 - 2013-05): theta = 0.2468 (SE 0.0957)
Seconda metà (2013-06 - 2026-10): theta = 0.2976 (SE 0.0988)
z = -0.369, p = 0.712  ->  parametro stabile = True


I due $\theta$ sono simili e la differenza non è significativa: il parametro è **stabile** nei due regimi di tassi.

## 9. Valutazione fuori campione

I controlli precedenti riguardano i dati usati per la stima. La prova decisiva è prevedere dati **nuovi**. Tengo da parte l'ultimo 20% della serie (64 mesi, da luglio 2021, un periodo con il forte rialzo dei tassi del 2022) e faccio previsioni **a un passo con origine mobile**: per ogni mese ristimo il modello su tutti i dati precedenti e prevedo il mese successivo.

Modelli confrontati: l'ARIMA(0, 1, 1) scelto, due alternative vicine (ARIMA(2, 1, 0), ARIMA(1, 1, 1)), l'ARIMA(15, 1, 1) della versione precedente e i modelli banali: random walk ($\hat y_{T+1} = y_T$) e random walk con drift.

Metriche: RMSE e MAE; **copertura** degli intervalli al 95% (quota di dati veri dentro l'intervallo, dovrebbe essere vicina al 95%); test di **Diebold-Mariano** per sapere se la differenza di errore quadratico tra due modelli è significativa:
$$d_t = e_{1,t}^2 - e_{2,t}^2, \qquad DM = \frac{\bar d}{\sqrt{\hat\sigma^2_{LR}(d)/n}} \sim N(0,1).$$

In [13]:
n_test = int(round(0.2 * T))
candidates = {
    "ARIMA(0,1,1)": lambda: ARIMA(0, 1, 1, const=False),
    "ARIMA(2,1,0)": lambda: ARIMA(2, 1, 0, const=False),
    "ARIMA(1,1,1)": lambda: ARIMA(1, 1, 1, const=False),
    "ARIMA(15,1,1)": lambda: ARIMA(15, 1, 1, const=False),
    "Random walk": lambda: RandomWalk(),
    "Random walk + drift": lambda: RandomWalk(drift=True),
}
oos, rows = {}, []
for name, build in candidates.items():
    actual, pred, err, lo, hi = rolling_one_step(build, y, n_test)
    oos[name] = (pred, err, lo, hi)
    rows.append([name, rmse(err), mae(err), coverage(actual, lo, hi)])
tab_oos = pd.DataFrame(rows, columns=["modello", "RMSE", "MAE", "copertura 95%"])
tab_oos["RMSE / RMSE RW"] = tab_oos["RMSE"] / tab_oos.loc[tab_oos["modello"] == "Random walk", "RMSE"].values[0]
display(tab_oos)

e_best = oos["ARIMA(0,1,1)"][1]
dm_rw = DieboldMariano_test(e_best, oos["Random walk"][1]).test()
dm_15 = DieboldMariano_test(e_best, oos["ARIMA(15,1,1)"][1]).test()
print("ARIMA(0,1,1) vs Random walk  :", dm_rw[3])
print("ARIMA(0,1,1) vs ARIMA(15,1,1):", dm_15[3])

test_dates = dates[T - n_test:]
pred, err, lo, hi = oos["ARIMA(0,1,1)"]
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(dates[T - n_test - 24:], y[T - n_test - 24:], color="black", label="Osservato")
ax.plot(test_dates, pred, color="tab:blue", label="Previsione a 1 passo ARIMA(0,1,1)")
ax.fill_between(test_dates, lo, hi, color="tab:blue", alpha=0.2, label="Intervallo al 95%")
ax.axvline(test_dates[0], color="grey", linestyle="--", linewidth=1)
ax.set_title("Previsioni fuori campione a un passo (origine mobile)")
ax.set_ylabel("%")
ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig("figures/fuori_campione.png", dpi=150)
plt.show()

R["n_test"], R["test_start"] = n_test, f"{test_dates[0]:%Y-%m}"
R["tab_oos"] = tab_oos.values.tolist()
R["dm_rw"], R["dm_15"] = list(dm_rw[:3]), list(dm_15[:3])

,modello,RMSE,MAE,copertura 95%,RMSE / RMSE RW
0,"ARIMA(0,1,1)",0.2086,0.1616,0.9531,0.9375
1,"ARIMA(2,1,0)",0.2087,0.1610,0.9688,0.9382
2,"ARIMA(1,1,1)",0.2093,0.1622,0.9531,0.9409
3,"ARIMA(15,1,1)",0.2188,0.1676,0.9375,0.9836
4,Random walk,0.2225,0.1733,0.9375,1.0000
5,Random walk + drift,0.2262,0.1758,0.9219,1.0168


ARIMA(0,1,1) vs Random walk  : DM = -1.780, p bilaterale = 0.075, p unilaterale (1 migliore di 2) = 0.038
ARIMA(0,1,1) vs ARIMA(15,1,1): DM = -1.930, p bilaterale = 0.054, p unilaterale (1 migliore di 2) = 0.027


- L'ARIMA(0, 1, 1) ha l'RMSE più basso, circa il **6% in meno del random walk**. Il miglioramento è modesto, come ci si aspetta per un tasso di mercato, ma sistematico: Diebold-Mariano dà $DM < 0$ con p-value unilaterale sotto il 5% (bilaterale 7,5%).
- ARIMA(2, 1, 0) e ARIMA(1, 1, 1) prevedono quasi come l'MA(1): confermano la stessa dinamica a un ritardo, ma con un parametro in più.
- L'ARIMA(15, 1, 1) della versione precedente prevede **peggio** dell'MA(1): i parametri in eccesso adattano il rumore del campione (sovradattamento).
- Il drift peggiora il random walk: coerente con il drift non significativo della sezione 2.1.
- La **copertura** degli intervalli dell'ARIMA(0, 1, 1) è vicina al 95%: nonostante le code spesse, gli intervalli a un passo sono affidabili in questo periodo.

## 10. Previsione a 12 mesi

Stimo il modello su tutto il campione e prevedo i prossimi 12 mesi. Per un ARIMA(0, 1, 1):
- $\hat y_{T+1} = y_T + \hat\theta\,\hat\varepsilon_T$, e da $h = 2$ in poi la previsione resta costante (gli shock futuri hanno media zero e non c'è drift);
- la varianza dell'errore a $h$ passi è $\hat\sigma^2\left[1 + (h-1)(1+\hat\theta)^2\right]$: cresce senza limite, come per ogni serie con radice unitaria.

In [14]:
H = 12
fc_mean, fc_var = model.forecast(H)
fc_se = np.sqrt(fc_var)
fc_dates = pd.date_range(dates[-1] + pd.offsets.MonthBegin(1), periods=H, freq="MS")
tab_fc = pd.DataFrame({"mese": fc_dates.strftime("%Y-%m"), "previsione": fc_mean, "SE": fc_se,
                       "inf 95%": fc_mean - 1.96 * fc_se, "sup 95%": fc_mean + 1.96 * fc_se})
display(tab_fc)

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(dates[-60:], y[-60:], color="black", label="Osservato")
ax.plot(fc_dates, fc_mean, color="tab:red", label="Previsione ARIMA(0,1,1)")
ax.fill_between(fc_dates, fc_mean - 1.96 * fc_se, fc_mean + 1.96 * fc_se, color="tab:red", alpha=0.2, label="Intervallo al 95%")
ax.set_title("Previsione a 12 mesi del rendimento a 10 anni")
ax.set_ylabel("%")
ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig("figures/previsione.png", dpi=150)
plt.show()

R["tab_fc"] = tab_fc.values.tolist()
R["eps_T"] = float(model.innovations[-1])

,mese,previsione,SE,inf 95%,sup 95%
0,2026-11,5.1448,0.2054,4.7422,5.5475
1,2026-12,5.1448,0.3378,4.4828,5.8069
2,2027-01,5.1448,0.4313,4.2995,5.9902
3,2027-02,5.1448,0.5079,4.1494,6.1402
4,2027-03,5.1448,0.5743,4.0192,6.2705
5,2027-04,5.1448,0.6338,3.9026,6.3871
6,2027-05,5.1448,0.6882,3.7959,6.4937
7,2027-06,5.1448,0.7386,3.6972,6.5925
8,2027-07,5.1448,0.7858,3.6047,6.6850
9,2027-08,5.1448,0.8303,3.5175,6.7722


La previsione è di poco sopra l'ultimo dato, perché l'ultimo shock è stato positivo e $\hat\theta>0$ ne trasmette una parte al mese successivo; poi resta piatta. L'intervallo si allarga rapidamente: già a un anno l'intervallo al 95% è circa $\pm 1{,}8$ punti percentuali attorno alla previsione. Con una radice unitaria e nessun drift, l'informazione utile per prevedere il livello è quasi tutta nell'ultimo valore osservato.

## 11. Controllo dell'implementazione con `statsmodels`

Le classi del progetto sono scritte da zero. Confronto i loro risultati con quelli di `statsmodels` sugli stessi dati: le statistiche dei test devono coincidere; le stime ARIMA differiscono leggermente perché `statsmodels` usa la massima verosimiglianza esatta invece di Hannan-Rissanen.

In [15]:
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import jarque_bera
from statsmodels.tsa.arima.model import ARIMA as SM_ARIMA
from statsmodels.tsa.stattools import adfuller, kpss
warnings.filterwarnings("ignore")

L_kpss = int(np.floor(4 * (T / 100) ** 0.25))
adf_mine = ADF_test(y, regression="ct"); adf_mine_stat = adf_mine.test()[0]
sm_fit = SM_ARIMA(y, order=(0, 1, 1), trend="n").fit()
sm_fc = sm_fit.get_forecast(H)

check = pd.DataFrame([
    ["ADF 'ct' su y", adf_mine_stat, adfuller(y, regression="ct", maxlag=adf_mine.lag, autolag=None)[0]],
    ["ADF 'c' su Δy", ADF_test(dy, regression="c").test()[0], adfuller(dy, regression="c", autolag="AIC")[0]],
    ["KPSS 'ct' su y", KPSS_test(y, regression="ct").test()[0], kpss(y, regression="ct", nlags=L_kpss)[0]],
    ["KPSS 'c' su Δy", KPSS_test(dy, regression="c").test()[0], kpss(dy, regression="c", nlags=L_kpss)[0]],
    ["Ljung-Box Δy, m=12", LjungBox_test(dy, 12).test()[0], acorr_ljungbox(dy, lags=[12])["lb_stat"].iloc[0]],
    ["Jarque-Bera Δy", JarqueBera_test(dy).test()[0], jarque_bera(dy)[0]],
    ["theta ARIMA(0,1,1)", theta, sm_fit.params[0]],
    ["SE theta", se_theta, sm_fit.bse[0]],
    ["sigma", np.sqrt(model.sigma2), np.sqrt(sm_fit.params[1])],
    ["previsione h=1", fc_mean[0], sm_fc.predicted_mean[0]],
    ["SE previsione h=12", fc_se[-1], np.sqrt(sm_fc.var_pred_mean[-1])],
], columns=["quantità", "progetto", "statsmodels"])
display(check)
R["check"] = check.values.tolist()

,quantità,progetto,statsmodels
0,ADF 'ct' su y,-1.4528,-1.4528
1,ADF 'c' su Δy,-12.3636,-12.3636
2,KPSS 'ct' su y,0.8619,0.8619
3,KPSS 'c' su Δy,0.4200,0.4200
4,"Ljung-Box Δy, m=12",27.4494,27.4494
5,Jarque-Bera Δy,83.1418,83.1418
6,"theta ARIMA(0,1,1)",0.3053,0.2950
7,SE theta,0.0621,0.0467
8,sigma,0.2054,0.2051
9,previsione h=1,5.1448,5.1453


Le statistiche dei test coincidono con `statsmodels`. Per l'ARIMA, $\hat\theta$ di Hannan-Rissanen (0,305) e di massima verosimiglianza (0,295) sono molto vicini e portano alle stesse conclusioni; l'errore standard di Hannan-Rissanen è più grande perché la seconda regressione usa meno osservazioni e shock stimati. Le previsioni coincidono alla terza cifra decimale.

## 12. Conclusioni

1. **Trasformazione.** GS10 è un tasso, non un prezzo: le variazioni in punti percentuali hanno varianza molto più stabile delle variazioni logaritmiche, quindi il modello è costruito sui livelli.
2. **Stazionarietà.** ADF e KPSS concordano: il livello ha una radice unitaria, le variazioni mensili sono stazionarie ($d = 1$). Il drift non è significativo: niente costante.
3. **Dinamica.** Le variazioni non sono rumore bianco (Ljung-Box). ACF e PACF indicano un MA(1), confermato da AIC e BIC. La dinamica ha una spiegazione precisa: i dati sono medie mensili di un tasso giornaliero, e la media di un random walk genera un MA(1) con $\theta = 2-\sqrt3 \approx 0{,}27$, compatibile con la stima.
4. **Bontà del modello.** $\theta$ significativo e stabile, modello invertibile, residui senza autocorrelazione. I residui hanno code spesse (Jarque-Bera), ma la copertura degli intervalli fuori campione è vicina al 95%.
5. **Previsione.** Fuori campione l'ARIMA(0, 1, 1) batte il random walk di circa il 6% in RMSE (Diebold-Mariano unilaterale significativo al 5%) e batte l'ARIMA(15, 1, 1) sovradattato della versione precedente.

**Limiti.** La varianza non è modellata (code spesse e periodi più volatili, come il 2022): un modello GARCH sarebbe il passo successivo, fuori dal programma del corso. Il guadagno predittivo dipende dalla media mensile: su dati giornalieri il tasso sarebbe molto più vicino a un random walk puro.

In [16]:
def to_py(o):
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, (list, tuple)): return [to_py(v) for v in o]
    if isinstance(o, dict): return {k: to_py(v) for k, v in o.items()}
    return o

with open("results.json", "w") as f:
    json.dump(to_py(R), f, indent=1)
print("Risultati salvati in results.json")

Risultati salvati in results.json
